# Fixed crop context and scale comparison

**CPU only. Run all four code cells. No training is used.**

This comparison uses the frozen PR158 reviewed-label checkpoint and the
same 20 development photographs. It first reproduces the current
full-image original/reflected method. Each of two fixed candidates then
adds the same two overlapping full-height crops, spanning columns
0–768 and 512–1280. Input 768 keeps the crop's original pixel scale.
This changes context and adds candidate exposure. Input 1280 also
enlarges those same crops by 5/3.

A crop box touching an artificial side boundary is excluded. A corner
outside its crop before native coordinate clipping is marked unavailable.
Every retained pose record is mapped to the original photograph. The
established score-ranked suppression rule selects whole records from the
baseline and both crops. It does not average new coordinates or tune scores.

The four references use fresh matching, with r67 primary. The return
includes misses, extras, duplicates, complete pairs, known corners on
partial fronts, empty photographs and every decision's source record.
Both candidate results are retained. This development comparison does
not select a deployment default.


## 1 Load the immutable source and fixed protocol

Source, verified inputs and outputs use separate directories. Existing
edited source is preserved. Choose a fresh `BAOMAI_CROP_CONTEXT_BASE` to
run beside it. The source modules, protocol and saved CPU control are
verified before inference.


In [1]:
from pathlib import Path
import hashlib, json, os, subprocess, sys

SOURCE_COMMIT = "5da7aa4f903dceb1e8300234ff98aebbdff06c17"
SOURCE_HASHES = {'scripts/analyze_crop_context_r71.py': '7befc607b653c3069a346898a88d9146e23c1a340c5cd603638947bc75f7d77a', 'scripts/diagnose_current_proposals_r69.py': '5e6e6a89e89e5f47adf513fa47dc79028895d0a06133cdbf685850cc1cec4b3b', 'scripts/analyze_partial_detection_r68.py': '18a2f0f35d051d87fb158c3a0908e461cf85c420b8f02b59ebb42574984f2d5a', 'scripts/analyze_duplicate_audit_r66.py': '868ab2c5df33ba376da4212aca6281d99a9d5c3a9d6cd1dac7dfaf4e1cd7acfa', 'scripts/analyze_proposal_stages_r63.py': '0dd585cb7d957dcd599c0f40fbbf8fcc1ffc85c9bf98f98307eb0c799f29f15b', 'scripts/audit_additional_front_pilot_r45_results.py': '4f2d5ddf167ae480ede653fa821efd8b36c5679aed9d3e79e98b1d7f71f4de3b', 'scripts/audit_additional_front_pilot_r47_results.py': '6d7a2d3ece42b9eb26a473437a8f8377474dee25d4b83c14a80cb04ab03d64d9', 'scripts/audit_front_diagnosis_r49_results.py': '5bb583f1cd9adfbf1cbef5084df935a6c2b2587f5b6c730b49ea26029d1244e2', 'scripts/audit_reflection_return.py': '55f391535b040a27682e899aec74915b87864063023bc1ed818f8f82e67101de', 'scripts/diagnose_fronts_r50.py': 'cba4f1df6dcf431c933af1506a4de15718636d9477406eb34779ac4d8ab6ee3e', 'scripts/evaluate_pose.py': '3f80bd09efbe97327b705ffe6ad3e662111f3e2bc71b69b51a9d38d7baa90ce0', 'scripts/export_front_error_review_r48.py': '45f8e98b2ecd77fc5cdd2db8debb018dfa3904cd373b875049eb491eb62227b5', 'scripts/pose_checkpoint.py': 'c1311307fd49b37e60f2939c9b084bc36690c1d13b6b31b9e4c51ea9353562d4', 'scripts/pose_scoring.py': '04e5cdbef26df978707d0b6e4b82f05471b6012845f7986b9c05c71644ec0018', 'scripts/predict_pose_folder.py': '2942864e86303a67911826fc0dbfe547b4b4fc2ed3860e74c9d0210c635520a6', 'scripts/replay_front_confidence_r51.py': '97828d8423f6b4f693a36c42bc23d1b868540bdfa6b75bfaf42373cffc5ba8c0', 'scripts/replay_partial_inclusion_r67.py': '5502022e8f5a7b510e875f2948d3dcbf4b1ab59addd94a9e5d3260ec3eda7f9a', 'scripts/run_corner_average_r58.py': '286232cf490f950b367068ea52b43078970ab27bb016169cb99159dab2fdbd88', 'scripts/run_flip_inference_r56.py': 'bf2ff9056f12976ca8c4aa971de2d2a7af1c679369c44dbf73de9f9d2542700d', 'scripts/run_inference_ablation_r53.py': '8c8a37c3dcaccde5c568e9df6f44651cdb05965ca5f5ed5b0f2978852387dcc1', 'scripts/run_original_comparison_r59.py': '279e2c0bd701299ea2a6f30403e97da68cc44a9c4ff874c5ee5a0d155a6e6ea3', 'scripts/run_reference_matrix_r60.py': '2939b2789abf2566d7680de60a0a87efae2a41e45c2dc01c9d15d1221cb8bbf1', 'scripts/run_training_label_comparison_r61.py': '0dc8d5dfdd344f590f88d17ea7ced76f3fc018026fae221a4df6e25d7c807a9c', 'scripts/analyze_corner_suppression_r69.py': '719ef2406a2d4a95311ff6ad866e161558a1b8e1b6632b3f60bd78b0c29e291f', 'scripts/build_crop_context_r71_notebook.py': '5f3a0d3b713df9214bc26708b267c783aa61c8442afe788ffedc2a643ace3a88', 'scripts/build_inference_ablation_r53_notebooks.py': 'e4acd00a9bfeb7334e90f1601f18ebe4d35688563991d761bb7748b3e38c39e7'}
FIXED_HASHES = {'docs/thesis/evidence/crop-context-r71/protocol.json': 'ae0dbd0e183aacb3f27c14b4691655c1e9f57823698188baa844c857291c42b7', 'docs/thesis/evidence/current-proposals-r69/native-predictions.json.gz': '66ffc3da1ade6d2522e098ce22bc3f0ae182c242dfa22259218283b33fae016c', 'docs/thesis/evidence/current-proposals-r69/predictions.json.gz': 'b069e017cb382f73edbf12a164b410766bf8e4051ded93217be221887523133c', 'docs/thesis/evidence/current-proposals-r69/per-image.json.gz': 'fbd184c0992bf4744ae7b2d7e2d7060840382c39d5adfdef8ab9abbfe6784b3f', 'docs/thesis/evidence/current-proposals-r69/summary.json': '973627f75a593f43ff3d35ca6f5924b8bcb125c0e7241b7446b4d8cf0bc379e7'}
BASE = Path(os.environ.get("BAOMAI_CROP_CONTEXT_BASE", "/content/baomai-crop-context-r71")).resolve()
SOURCE, WORK = BASE / "source", BASE / "work"
INPUT_DIR, OUTPUT = WORK / "inputs", WORK / "crop-context"
BASE.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0", "PYTHONDONTWRITEBYTECODE": "1"}
def run(command, cwd=None):
    subprocess.run(list(map(str, command)), cwd=cwd, env=ENV, check=True)
def git(*args):
    return subprocess.check_output(["git", *args], cwd=SOURCE, env=ENV, text=True).strip()
if not (SOURCE / ".git").exists():
    if SOURCE.exists() and any(SOURCE.iterdir()):
        raise RuntimeError("Existing files preserved. Choose a fresh BAOMAI_CROP_CONTEXT_BASE.")
    run(["git", "init", SOURCE])
    run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], SOURCE)
    run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_COMMIT], SOURCE)
    run(["git", "sparse-checkout", "init", "--no-cone"], SOURCE)
    run(["git", "sparse-checkout", "set", *['/scripts/', '/docs/thesis/evidence/partial-detection-r68/baomai-partial-detection-r68-results.zip', '/data/reviews/fronts-2026-11-28-r67/', '/docs/thesis/evidence/partial-inclusion-r67/excluded-front-provenance.json', '/docs/thesis/evidence/crop-context-r71/protocol.json', '/docs/thesis/evidence/current-proposals-r69/native-predictions.json.gz', '/docs/thesis/evidence/current-proposals-r69/predictions.json.gz', '/docs/thesis/evidence/current-proposals-r69/per-image.json.gz', '/docs/thesis/evidence/current-proposals-r69/summary.json']], SOURCE)
    run(["git", "checkout", "--detach", SOURCE_COMMIT], SOURCE)
def verify_source():
    if git("rev-parse", "HEAD") != SOURCE_COMMIT or git("status", "--porcelain", "--untracked-files=all"):
        raise RuntimeError("Source changed. Choose a fresh BAOMAI_CROP_CONTEXT_BASE.")
    for name, expected in {**SOURCE_HASHES, **FIXED_HASHES}.items():
        if hashlib.sha256((SOURCE / name).read_bytes()).hexdigest() != expected:
            raise RuntimeError("Pinned file differs: " + name)
verify_source()
WORK.mkdir(parents=True, exist_ok=True)
os.environ["YOLO_CONFIG_DIR"] = str(WORK / "runtime-config")
ENV["YOLO_CONFIG_DIR"] = os.environ["YOLO_CONFIG_DIR"]
print("Immutable source:", SOURCE_COMMIT)


Immutable source: 5da7aa4f903dceb1e8300234ff98aebbdff06c17


## 2 Verify the runtime, checkpoint, photographs and reference inputs

Existing Torch and torchvision installations are retained. CPU packages
are installed if absent. Ultralytics, NumPy, OpenCV and Pillow use the
recorded versions. Critical native source files are checked, and the
actual runtime is recorded. Reuse verified cached inputs or download
them from the immutable project paths.


In [2]:
from importlib.util import find_spec
from importlib.metadata import version, PackageNotFoundError
if find_spec("torch") is None or find_spec("torchvision") is None:
    run([sys.executable, "-m", "pip", "install", "--quiet", "--index-url",
         "https://download.pytorch.org/whl/cpu", "torch==2.10.0+cpu", "torchvision==0.25.0+cpu"])
required = {"ultralytics": "8.4.182", "numpy": "2.4.6", "opencv-python": "5.0.0.93", "Pillow": "13.0.0"}
def installed(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None
missing = [name + "==" + expected for name, expected in required.items() if installed(name) != expected]
if missing:
    constraints = WORK / "torch-constraints.txt"
    constraints.write_text("".join(name + "==" + version(name) + "\n" for name in ("torch", "torchvision")))
    run([sys.executable, "-m", "pip", "install", "--quiet", "--constraint", constraints, *missing])
sys.dont_write_bytecode = True
sys.path.insert(0, str(SOURCE / "scripts"))
import analyze_crop_context_r71 as study
plan = study.prepare(SOURCE, INPUT_DIR)
runtime = study.current.native_runtime()
print("Frozen checkpoint:", plan["checkpoint"])
print("Verified photographs:", len(plan["images"]))
print("Runtime packages:", runtime["packages"])


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/workspace/scratch/be67d3adbd23/r71-notebook-run/work/runtime-config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Frozen checkpoint: {'filename': 'reviewed_labels-last.pt', 'bytes': 21704814, 'sha256': 'c749fcff2126467cbc770f75a71fd79da51ebd08308ba909a448c3dd8cb5b372'}
Verified photographs: 20
Runtime packages: {'torch': '2.10.0+cpu', 'torchvision': '0.25.0+cpu', 'ultralytics': '8.4.182', 'numpy': '2.4.6', 'opencv-python': '5.0.0.93', 'Pillow': '13.0.0'}


## 3 Verify the control and run both fixed candidates

The fresh full-image control must reproduce saved R69 CPU per-image
matches, misses, extra and duplicate identities, and every known-corner
five- and ten-pixel outcome. Numeric record differences are retained.
A failed control stops the run before either crop candidate.

The run makes 40 full-image calls and 80 crop calls. It records exact
letterbox geometry, pre-clamp crop corners, artificial-edge decisions,
source IDs, merged selections and fresh evaluations. The checkpoint and
frozen model state must remain unchanged. All-negative photographs are
included in each arm.


In [3]:
summary, records = study.analyze(SOURCE, INPUT_DIR, OUTPUT, plan)
study.write_outputs(SOURCE, OUTPUT, summary, records)
from IPython.display import Markdown, display
display(Markdown((OUTPUT / "results.md").read_text()))
print("Control gate:", summary["control_gate_passed"])
print("Recorded comparisons:", summary["record_counts"])
print("Crop decisions:", json.dumps(summary["crop_counts"], indent=2))


Control gate: True
Recorded comparisons: {'forward_calls': 120, 'baseline_forward_calls': 40, 'crop_forward_calls': 80, 'matrix_rows': 12, 'paired_targets': 5284}
Crop decisions: {
  "crop_context_768": {
    "native_outputs": 1149,
    "boundary_rejected": 193,
    "pooled_crop_outputs": 956,
    "invalidated_points_all_native_outputs": 184,
    "invalidated_points_pooled_outputs": 8,
    "retained_baseline_outputs": 309,
    "retained_crop_outputs": 383
  },
  "crop_context_scale_1280": {
    "native_outputs": 1145,
    "boundary_rejected": 189,
    "pooled_crop_outputs": 956,
    "invalidated_points_all_native_outputs": 148,
    "invalidated_points_pooled_outputs": 2,
    "retained_baseline_outputs": 396,
    "retained_crop_outputs": 351
  }
}


# Fixed crop context and scale comparison

The frozen reviewed-label model was run on the same 20 development photographs. The fresh full-image control passed the saved CPU matching and corner-outcome checks.

Each candidate adds the same two overlapping full-height crops. The first keeps the original image scale. The second enlarges each crop by 5/3. Artificial-edge boxes are rejected before whole-record suppression.

The first candidate changes context and adds candidate exposure. The second adds enlargement to the same crop geometry.

| Reference | Method | Missed | Extras | Duplicates | Both corners within 5 px | Both within 10 px | Empty-scene outputs |
|---|---|---:|---:|---:|---:|---:|---:|
| r50 | baseline | 15 | 51 | 14 | 572/583 | 583/583 | 0 |
| r67 | baseline | 18 | 50 | 14 | 572/583 | 583/583 | 0 |
| supplied664 | baseline | 50 | 76 | 12 | 513/584 | 566/584 | 0 |
| frozen666 | baseline | 38 | 62 | 14 | 523/584 | 575/584 | 0 |
| r50 | crop_context_768 | 14 | 52 | 11 | 572/583 | 581/583 | 0 |
| r67 | crop_context_768 | 17 | 51 | 11 | 572/583 | 581/583 | 0 |
| supplied664 | crop_context_768 | 49 | 77 | 11 | 507/584 | 563/584 | 0 |
| frozen666 | crop_context_768 | 37 | 63 | 13 | 517/584 | 572/584 | 0 |
| r50 | crop_context_scale_1280 | 14 | 107 | 44 | 566/583 | 581/583 | 0 |
| r67 | crop_context_scale_1280 | 17 | 106 | 44 | 566/583 | 581/583 | 0 |
| supplied664 | crop_context_scale_1280 | 46 | 129 | 38 | 506/584 | 565/584 | 0 |
| frozen666 | crop_context_scale_1280 | 34 | 115 | 40 | 516/584 | 574/584 | 0 |

## Paired changes under r67

- crop_context_768: 1 recovered fronts and 0 newly missed fronts. Five-pixel pair gains/losses: 1/1. Ten-pixel pair gains/losses: 0/2.
- crop_context_scale_1280: 2 recovered fronts and 1 newly missed fronts. Five-pixel pair gains/losses: 4/10. Ten-pixel pair gains/losses: 0/2.

The package contains every native crop record, pre-clamp corner coordinate, boundary decision, mapped point, pooled source, suppression decision and paired target outcome. All four references use fresh matching. This comparison does not train a model or select a deployment default.


## 4 Verify and download the evidence

Recheck the immutable source and every saved member, then package the
complete predictions and decision records. The archive contains both
fixed candidate outcomes, the protocol, runtime and input identities.
It contains no new model or training labels.


In [4]:
verify_source()
RESULT = study.package_results(OUTPUT, SOURCE_COMMIT)
manifest = {"source_commit": SOURCE_COMMIT, "notebook_code_cells": 4,
            "fresh_inference": True, "training": False, "optimizer_updates": 0,
            "labels_changed": False, "threshold_search": False, "automatic_promotion": False,
            "control_gate_passed": summary["control_gate_passed"],
            "matrix_rows": len(summary["matrix"]),
            "result": {"filename": RESULT.name, "bytes": RESULT.stat().st_size,
                       "sha256": hashlib.sha256(RESULT.read_bytes()).hexdigest()}}
(WORK / "package-manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(json.dumps(manifest, indent=2))
from IPython.display import FileLink
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(RESULT)))
else:
    files.download(str(RESULT))


{
  "source_commit": "5da7aa4f903dceb1e8300234ff98aebbdff06c17",
  "notebook_code_cells": 4,
  "fresh_inference": true,
  "training": false,
  "optimizer_updates": 0,
  "labels_changed": false,
  "threshold_search": false,
  "automatic_promotion": false,
  "control_gate_passed": true,
  "matrix_rows": 12,
  "result": {
    "filename": "baomai-crop-context-r71-results.zip",
    "bytes": 3322927,
    "sha256": "99e18b001b0107d0728802042616548f9045cc8b2d8ec041ea22e3dc67e47c9d"
  }
}


/workspace/scratch/be67d3adbd23/r71-notebook-run/work/crop-context/baomai-crop-context-r71-results.zip